In order to run the following notebooks, if you haven't done yet, you need to set the openai key inside .env file as `OPENAI_API_KEY`

In [18]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("QWEN_API_KEY","")
assert API_KEY, "ERROR: DEEPSEEK Key is missing"

client = OpenAI(
    api_key=API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
    )
model = 'text-embedding-v4'

SIMILARITIES_RESULTS_THRESHOLD = 0.75
DATASET_NAME = "../embedding_index_3m.json"

Next, we are going to load the Embedding Index into a Pandas Dataframe. The Embedding Index is stored in a JSON file called `embedding_index_3m.json`. The Embedding Index contains the Embeddings for each of the YouTube transcripts up until late Oct 2023.

In [19]:
def load_dataset(source: str) -> pd.core.frame.DataFrame:
    # Load the video session index
    pd_vectors = pd.read_json(source)
    return pd_vectors.drop(columns=["text"], errors="ignore").fillna("")

In [17]:
import json
from tqdm import tqdm  # 进度条（可选）

# 假设你已经有 client（Qwen / 通义）
# model 名字按你实际用的改
MODEL_NAME = "text-embedding-v4"

# 读取原始 JSON
with open(DATASET_NAME, "r", encoding="utf-8") as f:
    data = json.load(f)

# 遍历每一条数据
for item in tqdm(data):
    text = item.get("summary", "")

    if not text:
        item["embedding_qwen"] = None
        continue

    # 调用 embedding API
    response = client.embeddings.create(
        input=text,
        model=MODEL_NAME
    )

    embedding = response.data[0].embedding

    # 保存新 embedding
    item["embedding_qwen"] = embedding

# 保存新文件（推荐另存，不要覆盖原文件）
with open("../data_with_qwen_embedding.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

  1%|          | 14/1409 [00:03<05:54,  3.94it/s]


KeyboardInterrupt: 

Next, we are going to create a function called `get_videos` that will search the Embedding Index for the query. The function will return the top 5 videos that are most similar to the query. The function works as follows:

1. First, a copy of the Embedding Index is created.
2. Next, the Embedding for the query is calculated using the OpenAI Embedding API.
3. Then a new column is created in the Embedding Index called `similarity`. The `similarity` column contains the cosine similarity between the query Embedding and the Embedding for each video segment.
4. Next, the Embedding Index is filtered by the `similarity` column. The Embedding Index is filtered to only include videos that have a cosine similarity greater than or equal to 0.75.
5. Finally, the Embedding Index is sorted by the `similarity` column and the top 5 videos are returned.

In [23]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def get_videos(
    query: str, dataset: pd.core.frame.DataFrame, rows: int
) -> pd.core.frame.DataFrame:
    # create a copy of the dataset
    video_vectors = dataset.copy()

    # get the embeddings for the query    
    query_embeddings = client.embeddings.create(input=query, model=model).data[0].embedding

    # create a new column with the calculated similarity for each row
    video_vectors["similarity"] = video_vectors["embedding_qwen"].apply(
    lambda x: cosine_similarity(np.array(query_embeddings), np.array(x))
)

    # filter the videos by similarity
    mask = video_vectors["similarity"] >= SIMILARITIES_RESULTS_THRESHOLD
    video_vectors = video_vectors[mask].copy()

    # sort the videos by similarity
    video_vectors = video_vectors.sort_values(by="similarity", ascending=False).head(
        rows
    )

    # return the top rows
    return video_vectors.head(rows)

This function is very simple, it just prints out the results of the search query.

In [24]:
def display_results(videos: pd.core.frame.DataFrame, query: str):
    def _gen_yt_url(video_id: str, seconds: int) -> str:
        """convert time in format 00:00:00 to seconds"""
        return f"https://youtu.be/{video_id}?t={seconds}"

    print(f"\nVideos similar to '{query}':")
    for _, row in videos.iterrows():
        youtube_url = _gen_yt_url(row["videoId"], row["seconds"])
        print(f" - {row['title']}")
        print(f"   Summary: {' '.join(row['summary'].split()[:15])}...")
        print(f"   YouTube: {youtube_url}")
        print(f"   Similarity: {row['similarity']}")
        print(f"   Speakers: {row['speaker']}")

1. First, the Embedding Index is loaded into a Pandas Dataframe.
2. Next, the user is prompted to enter a query.
3. Then the `get_videos` function is called to search the Embedding Index for the query.
4. Finally, the `display_results` function is called to display the results to the user.
5. The user is then prompted to enter another query. This process continues until the user enters `exit`.

![](../images/notebook-search.png?WT.mc_id=academic-105485-koreyst)

You will be prompted to enter a query. Enter a query and press enter. The application will return a list of videos that are relevant to the query. The application will also return a link to the place in the video where the answer to the question is located.

Here are some queries to try out:

- What is Azure Machine Learning?
- How do convolutional neural networks work?
- What is a neural network?
- Can I use Jupyter Notebooks with Azure Machine Learning?
- What is ONNX?

In [25]:
pd_vectors = load_dataset("../data_with_qwen_embedding.json")

# get user query from input
while True:
    query = input("Enter a query: ")
    if query == "exit":
        break
    videos = get_videos(query, pd_vectors, 5)
    display_results(videos, query)


Videos similar to ' What is Azure Machine Learning?':
 - OSS Framework Support in Azure Machine Learning Service
   Summary: Azure Machine Learning supports MLflow, an open-source platform that helps manage the lifecycle of machine...
   YouTube: https://youtu.be/zH1UojWq0oU?t=549
   Similarity: 0.8065705999842983
   Speakers: Shivani Patel, Andy
 - AI Show | Nov 5 | Ignite Recap | Arc Enabled ML
   Summary: Azure Arc-enabled machine learning is a solution that addresses the needs of customers who require...
   YouTube: https://youtu.be/yl_g-HhGVGI?t=184
   Similarity: 0.7807047191567854
   Speakers: Bozhong Lin, Bea Stollnitz
 - What’s new with Azure Machine Learning
   Summary: Azure Machine Learning is a powerful tool for training and deploying machine learning models. It...
   YouTube: https://youtu.be/YlWCeY_CWEg?t=914
   Similarity: 0.7788527687175502
   Speakers: Chris
 - Build “zero code” machine learning models with Azure Machine Learning service
   Summary: Eduardo Mellow, a